In [ ]:
# Cell 1) Install
!pip install -q transformers datasets accelerate scikit-learn

import os
import re
import time
import math
import random
import numpy as np
import torch
import torch.nn as nn

from dataclasses import dataclass
from typing import List, Tuple, Dict, Any, Optional

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    ElectraModel,
    ElectraPreTrainedModel,
)
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix

print("torch:", torch.__version__)
import transformers
print("transformers:", transformers.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


torch: 2.9.0+cu126
transformers: 4.57.6
device: cuda


In [ ]:
# Cell 2) Paths
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2"

TRAIN_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_train.Y.txt")
VAL_Y_PATH   = os.path.join(BASE_PATH, "iwslt2017_en_validation.Y.txt")
TEST_Y_PATH  = os.path.join(BASE_PATH, "iwslt2017_en_test.Y.txt")

for p in [TRAIN_Y_PATH, VAL_Y_PATH, TEST_Y_PATH]:
    assert os.path.exists(p), f"Not found: {p}"

def load_lines(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_y = load_lines(TRAIN_Y_PATH)
val_y   = load_lines(VAL_Y_PATH)
test_y  = load_lines(TEST_Y_PATH)

print("train/val/test:", len(train_y), len(val_y), len(test_y))
print("example Y:", val_y[0])


Mounted at /content/drive
train/val/test: 357117 1501 10799
example Y: Last year I showed these two slides so that demonstrate that the arctic ice cap, which for most of the last three million years has been the size of the lower 48 states, has shrunk by 40 percent.


In [ ]:
# Cell 3) Labels + Y -> (words, labels) parser
LABEL2ID = {"O": 0, "COMMA": 1, "PERIOD": 2, "QMARK": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

POLACEK_W = np.array([1.0, 1.5, 2.0, 5.0], dtype=np.float32)

def _strip_trailing_closers(s: str) -> str:
    return re.sub(r'[\)\]\}\\"\'”’]+$', '', s)

def _extract_punct_label_from_token(tok: str) -> Tuple[str, str]:
    """
    Returns (clean_word, punct_label_str)
    punct_label corresponds to punctuation that should appear AFTER this word.
    """
    t = tok.strip()
    if not t:
        return "", "O"

    t2 = _strip_trailing_closers(t)

    punct = None
    if t2.endswith(","):
        punct = "COMMA"
        word = t2[:-1]
    elif t2.endswith("."):
        punct = "PERIOD"
        word = t2[:-1]
    elif t2.endswith("?"):
        punct = "QMARK"
        word = t2[:-1]
    else:
        punct = "O"
        word = t2

    # ( { delete
    word = re.sub(r'^[\(\[\{\\"\'“‘]+', '', word).strip()

    # If word becomes empty drop it
    if not word:
        return "", "O"

    return word, punct

def parse_y_to_words_labels(y_sentence: str) -> Tuple[List[str], List[int]]:
    toks = y_sentence.split()
    words = []
    labels = []
    for tok in toks:
        w, p = _extract_punct_label_from_token(tok)
        if not w:
            continue
        words.append(w)
        labels.append(LABEL2ID[p])
    return words, labels

# quick sanity check
w0, l0 = parse_y_to_words_labels(val_y[0])
print("Y:", val_y[0])
print("words:", w0[:40])
print("labels:", [ID2LABEL[x] for x in l0[:40]])
print("X:", " ".join(w0[:40]))
print(f"'O' -> {LABEL2ID['O']}")
print(f"'COMMA' -> {LABEL2ID['COMMA']}")
print(f"'PERIOD' -> {LABEL2ID['PERIOD']}")
print(f"'QMARK' -> {LABEL2ID['QMARK']}")


Y: Last year I showed these two slides so that demonstrate that the arctic ice cap, which for most of the last three million years has been the size of the lower 48 states, has shrunk by 40 percent.
words: ['Last', 'year', 'I', 'showed', 'these', 'two', 'slides', 'so', 'that', 'demonstrate', 'that', 'the', 'arctic', 'ice', 'cap', 'which', 'for', 'most', 'of', 'the', 'last', 'three', 'million', 'years', 'has', 'been', 'the', 'size', 'of', 'the', 'lower', '48', 'states', 'has', 'shrunk', 'by', '40', 'percent']
labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'COMMA', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'COMMA', 'O', 'O', 'O', 'O', 'PERIOD']
X: Last year I showed these two slides so that demonstrate that the arctic ice cap which for most of the last three million years has been the size of the lower 48 states has shrunk by 40 percent
'O' -> 0
'COMMA' -> 1
'PERIOD' -> 2
'QMARK' -> 3


In [ ]:
# Cell 4) Tokenize
MODEL_NAME = "google/electra-small-discriminator"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

MAX_LEN = 256

def tokenize_and_align(words: List[str], word_labels: List[int]) -> Dict[str, Any]:
    enc = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
        return_attention_mask=True,
    )

    word_ids = enc.word_ids()
    aligned = [-100] * len(enc["input_ids"])

    #exp) tokens: [CLS] And you sa ##y , [SEP]
    #     word_ids : [None, 0, 1, 2, 2, None]

    # last token position for each word id
    last_pos_for_word = {}
    for tpos, wid in enumerate(word_ids):
        if wid is None:
            continue
        last_pos_for_word[wid] = tpos

    for wid, tpos in last_pos_for_word.items():
        if wid < len(word_labels):
            aligned[tpos] = word_labels[wid]

    enc["labels"] = aligned
    return enc

def build_dataset(y_list: List[str]) -> Tuple[Dataset, List[List[str]], List[List[int]]]:
    rows = []
    all_words = []
    all_labels = []
    for y in y_list:
        words, labels = parse_y_to_words_labels(y)
        all_words.append(words)
        all_labels.append(labels)
        rows.append(tokenize_and_align(words, labels))
    return Dataset.from_list(rows), all_words, all_labels

train_ds, train_words, train_labels = build_dataset(train_y)
val_ds,   val_words,   val_labels   = build_dataset(val_y)
test_ds,  test_words,  test_labels  = build_dataset(test_y)

print(train_ds[0].keys())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


In [ ]:
# Cell 5) Polacek-style model head: 256 -> 512 (SELU) -> 4

class ElectraPolacekForTokenClassification(ElectraPreTrainedModel):
    def __init__(self, config, num_labels=4):
        super().__init__(config)
        self.num_labels = num_labels
        self.electra = ElectraModel(config)
        hidden = config.hidden_size  # electra-small=256

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.SELU(),
            nn.Linear(512, num_labels),
        )
        self.post_init()


    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        sequence_output = outputs.last_hidden_state  # [B, T, H] (Batch size, sequence length, hidden size)
        logits = self.classifier(sequence_output)    # [B, T, C]

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits}

config = AutoConfig.from_pretrained(MODEL_NAME)
model = ElectraPolacekForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    num_labels=4,
)

model.to(device)
print("model ready")


pytorch_model.bin:   0%|          | 0.00/54.2M [00:00<?, ?B/s]

Some weights of ElectraPolacekForTokenClassification were not initialized from the model checkpoint at google/electra-small-discriminator and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model ready


In [ ]:
# Cell 6) TrainingArguments (compat with transformers version changes)
def make_training_args(output_dir: str):
    # evaluation_strategy(x) eval_strategy (O)
    kwargs_common = dict(
        output_dir=output_dir,
        learning_rate=3e-5,  # backbone lr (we will override with param groups in optimizer)
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1_4class",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    try:
        return TrainingArguments(
            **kwargs_common,
            evaluation_strategy="epoch",
            save_strategy="epoch",
        )
    except TypeError:
        return TrainingArguments(
            **kwargs_common,
            eval_strategy="epoch",
            save_strategy="epoch",
        )

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def apply_polacek_weighting_probs(logits_np: np.ndarray) -> np.ndarray:
    x = logits_np - logits_np.max(axis=-1, keepdims=True)
    p = np.exp(x)
    p = p / p.sum(axis=-1, keepdims=True)
    p = p * POLACEK_W[None, ...]
    p = p / p.sum(axis=-1, keepdims=True)
    return p

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # logits: [B, T, C], labels: [B, T]
    preds = np.argmax(logits, axis=-1)

    y_true = []
    y_pred = []
    for pr, lb in zip(preds, labels):
        for p, y in zip(pr, lb):
            if y == -100:
                continue
            y_true.append(int(y))
            y_pred.append(int(p))

    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0,1,2,3], average=None, zero_division=0
    )

    macro_f1_4 = float(np.mean(f1))
    punct_ids = [1,2,3]
    _, _, punct_macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=punct_ids, average="macro", zero_division=0
    )

    y_true_bin = [0 if y == 0 else 1 for y in y_true]
    y_pred_bin = [0 if y == 0 else 1 for y in y_pred]
    _, _, punct_binary_f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average="binary", zero_division=0
    )

    return {
        "macro_f1_4class": macro_f1_4,
        "punct_macro_f1": float(punct_macro_f1),
        "punct_binary_f1": float(punct_binary_f1),
        "f1_O": float(f1[0]),
        "f1_COMMA": float(f1[1]),
        "f1_PERIOD": float(f1[2]),
        "f1_QMARK": float(f1[3]),
    }

def print_confusion_matrix(
    y_true,
    y_pred,
    labels=(0, 1, 2, 3),
    label_names=("O", "COMMA", "PERIOD", "QMARK"),
):
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))

    header = [""] + [f"Pred_{label_names[i]}" for i in range(len(labels))]
    print("\nConfusion Matrix")
    print("  ".join(f"{h:>12s}" for h in header))

    for r, lab in enumerate(labels):
        row = [f"True_{label_names[r]}"] + [str(int(x)) for x in cm[r]]
        print("  ".join(f"{c:>12s}" for c in row))

    return cm

In [ ]:
# Cell 7) Polacek optimizer settings: backbone lr=3e-5, head lr=1e-4
# Also decay after epoch2: "starting from the second epoch, lr decayed by 0.95 every 5000 steps"
# We approximate this with LambdaLR based on global step and only apply decay after half of training steps.

from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

class PolacekTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer

        base_params = []
        head_params = []

        for n, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            if n.startswith("electra."):
                base_params.append(p)
            else:
                head_params.append(p)

        self.optimizer = AdamW(
            [
                {"params": base_params, "lr": 3e-5},
                {"params": head_params, "lr": 1e-4},
            ],
            weight_decay=0.01,
        )
        return self.optimizer

    def create_scheduler(self, num_training_steps: int, optimizer=None):
        if self.lr_scheduler is not None:
            return self.lr_scheduler
        if optimizer is None:
            optimizer = self.optimizer

        # decay starts from second epoch (after half steps when epochs=2)
        half = max(1, num_training_steps // 2)
        step_unit = 5000
        gamma = 0.95

        def lr_lambda(current_step: int):
            if current_step < half:
                return 1.0
            k = (current_step - half) // step_unit
            return (gamma ** k)

        self.lr_scheduler = LambdaLR(optimizer, lr_lambda)
        return self.lr_scheduler

training_args = make_training_args(output_dir="/content/drive/MyDrive/experiments/electra_polacek_baseline")

trainer = PolacekTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-350257353.py:55: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `PolacekTrainer.__init__`. Use `processing_class` instead.
  trainer = PolacekTrainer(


In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/experiments/electra_polacek_baseline"

import os
from glob import glob
if not os.path.exists(os.path.join(CHECKPOINT_PATH, "pytorch_model.bin")):
    sub_folders = glob(os.path.join(CHECKPOINT_PATH, "checkpoint-*"))
    if sub_folders:
        CHECKPOINT_PATH = sorted(sub_folders)[-1]

print(f"Loading weights from: {CHECKPOINT_PATH}")

model = ElectraPolacekForTokenClassification.from_pretrained(CHECKPOINT_PATH)
model.to(device)
model.eval()

Loading weights from: /content/drive/MyDrive/experiments/electra_polacek_baseline/checkpoint-44640


ElectraPolacekForTokenClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense)

In [ ]:
#2-subword token lookahead

import time
import numpy as np
import torch
from typing import List, Optional

@torch.no_grad()
def predict_streaming_right_tokens_one_sentence(
    model,
    tokenizer,
    words: List[str],
    K: int = 2,           # right context: K subword tokens
    max_len: int = 256,
    device: str = "cuda",
):
    model.eval()

    # full tokenization once (no truncation)
    full = tokenizer(
        words,
        is_split_into_words=True,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    full_ids = full["input_ids"]              # includes [CLS] ... [SEP]
    word_ids = full.word_ids()                # token -> word index (None for specials)

    # find SEP position
    try:
        sep_id = tokenizer.sep_token_id
        sep_pos = full_ids.index(sep_id)
    except Exception:
        # fallback: assume last token is SEP
        sep_pos = len(full_ids) - 1

    # last subword token position for each word
    last_pos = {}
    for tpos, wid in enumerate(word_ids):
        if wid is None:
            continue
        last_pos[wid] = tpos

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    out_labels = []

    # token range excluding specials: 1 .. sep_pos-1
    min_core = 1
    max_core = max(1, sep_pos - 1)

    core_budget = max_len - 2  # reserve for [CLS], [SEP]
    if core_budget < 1:
        raise ValueError("max_len too small")

    for i in range(len(words)):
        if i not in last_pos:
            out_labels.append(0)
            continue

        t_i = last_pos[i]  # absolute pos in full_ids

        # right boundary in core tokens: include K tokens to the right of t_i (excluding SEP)
        r = min(t_i + K, max_core)

        # choose l so that window core length <= core_budget, and include target
        l = max(min_core, r - core_budget + 1)
        if l > t_i:
            l = t_i  # ensure target is inside

        # build window ids with specials
        core_ids = full_ids[l : r + 1]
        window_ids = [cls_id] + core_ids + [sep_id]
        attention_mask = [1] * len(window_ids)
        token_type_ids = [0] * len(window_ids)

        input_ids = torch.tensor([window_ids], device=device)
        attention_mask = torch.tensor([attention_mask], device=device)
        token_type_ids = torch.tensor([token_type_ids], device=device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        logits = out["logits"][0]  # [T, C]

        # position of target token inside window (+1 because CLS)
        t_in_window = (t_i - l) + 1
        if t_in_window < 0 or t_in_window >= logits.shape[0]:
            out_labels.append(0)
            continue

        logit_i = logits[t_in_window].detach().cpu().numpy()  # [C]

        # Polacek weighting (you already have apply_polacek_weighting_probs)
        prob = apply_polacek_weighting_probs(logit_i[None, :])[0]
        pred = int(np.argmax(prob))
        out_labels.append(pred)

    return out_labels


def evaluate_streaming_right_tokens(
    model,
    tokenizer,
    words_list: List[List[str]],
    labels_list: List[List[int]],
    K: int = 2,
    max_len: int = 256,
    device: str = "cuda",
    max_sentences: Optional[int] = None,
):
    from sklearn.metrics import classification_report

    y_true, y_pred = [], []
    n_sent, n_words = 0, 0
    t0 = time.time()

    for words, labels in zip(words_list, labels_list):
        if max_sentences is not None and n_sent >= max_sentences:
            break

        pred = predict_streaming_right_tokens_one_sentence(
            model, tokenizer, words, K=K, max_len=max_len, device=device
        )

        y_true.extend(labels)
        y_pred.extend(pred)
        n_sent += 1
        n_words += len(words)

    t1 = time.time()
    elapsed = t1 - t0
    print(f"STREAMING right_tokens={K} | sentences={n_sent} | words={n_words} | elapsed_sec={elapsed:.3f}")
    if elapsed > 0:
        print("words_per_sec:", n_words / elapsed)

    print(classification_report(
        y_true, y_pred,
        target_names=["O","COMMA","PERIOD","QMARK"],
        digits=4,
        zero_division=0
    ))

    print_confusion_matrix(y_true, y_pred)



In [ ]:
from sklearn.metrics import classification_report
from tqdm import tqdm
import torch
import numpy as np

target_dataset = val_y[:100]

all_gold_ids = []
all_pred_ids = []

model.eval()

for i, y_true_text in enumerate(tqdm(target_dataset)):
    try:
        words, gold_ids = parse_y_to_words_labels(y_true_text)

        if not words: continue

        # (K=2)
        pred_ids = predict_streaming_right_tokens_one_sentence(
            model,
            tokenizer,
            words,
            K=2,
            max_len=256,
            device=device
        )

        min_len = min(len(gold_ids), len(pred_ids))
        all_gold_ids.extend(gold_ids[:min_len])
        all_pred_ids.extend(pred_ids[:min_len])

    except Exception as e:
        print(f"Error at index {i}: {e}")
        continue

print("\n" + "="*50)
print(f"   [Result on {NUM_SAMPLES} Samples (K=2)]")
print("="*50)

target_names = ["O", "COMMA", "PERIOD", "QMARK"]
print(classification_report(all_gold_ids, all_pred_ids, labels=[0, 1, 2, 3], target_names=target_names, digits=3))


print_confusion_matrix(all_gold_ids, all_pred_ids)

100%|██████████| 100/100 [00:12<00:00,  7.89it/s]


   [Result on 100 Samples (K=2)]
              precision    recall  f1-score   support

           O     0.9835    0.9844    0.9839      1088
       COMMA     0.7213    0.7097    0.7154        62
      PERIOD     1.0000    1.0000    1.0000        99
       QMARK     0.0000    0.0000    0.0000         0

    accuracy                         0.9720      1249
   macro avg     0.6762    0.6735    0.6748      1249
weighted avg     0.9718    0.9720    0.9719      1249


Confusion Matrix
                    Pred_O    Pred_COMMA   Pred_PERIOD    Pred_QMARK
      True_O          1071            17             0             0
  True_COMMA            18            44             0             0
 True_PERIOD             0             0            99             0
  True_QMARK             0             0             0             0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

array([[1071,   17,    0,    0],
       [  18,   44,    0,    0],
       [   0,    0,   99,    0],
       [   0,    0,    0,    0]])

In [ ]:
from sklearn.metrics import classification_report
from tqdm import tqdm
import torch
import numpy as np

target_dataset = test_y

all_gold_ids = []
all_pred_ids = []

model.eval()

for i, y_true_text in enumerate(tqdm(target_dataset)):
    try:

        words, gold_ids = parse_y_to_words_labels(y_true_text)

        if not words: continue


        pred_ids = predict_streaming_right_tokens_one_sentence(
            model,
            tokenizer,
            words,
            K=2,
            max_len=256,
            device=device
        )

        min_len = min(len(gold_ids), len(pred_ids))
        all_gold_ids.extend(gold_ids[:min_len])
        all_pred_ids.extend(pred_ids[:min_len])

    except Exception as e:
        print(f"Error at index {i}: {e}")
        continue

print("\n" + "="*50)
print(f"   [Result on {NUM_SAMPLES} Samples (K=2)]")
print("="*50)

target_names = ["O", "COMMA", "PERIOD", "QMARK"]
print(classification_report(all_gold_ids, all_pred_ids, labels=[0, 1, 2, 3], target_names=target_names, digits=3))

print_confusion_matrix(all_gold_ids, all_pred_ids)

100%|██████████| 10799/10799 [30:42<00:00,  5.86it/s]


   [Result on 100 Samples (K=2)]
              precision    recall  f1-score   support

           O      0.977     0.985     0.981    160196
       COMMA      0.797     0.713     0.752     12999
      PERIOD      0.992     0.992     0.992     10172
       QMARK      0.933     0.917     0.925       911

    accuracy                          0.966    184278
   macro avg      0.925     0.902     0.913    184278
weighted avg      0.965     0.966     0.965    184278


Confusion Matrix
                    Pred_O    Pred_COMMA   Pred_PERIOD    Pred_QMARK
      True_O        157831          2348            10             7
  True_COMMA          3732          9266             0             1
 True_PERIOD            13            15         10092            52
  True_QMARK             1             2            73           835


array([[157831,   2348,     10,      7],
       [  3732,   9266,      0,      1],
       [    13,     15,  10092,     52],
       [     1,      2,     73,    835]])